# 01. Data Preprocessing & Leakage Free Split

Prepare clinical_data.csv for multi-class (`NEV`, `ACK`, `SEK`, `BCC`, `SCC`, `MEL`) and binary (`is_cancer`) classification with strict `patient_id` grouping.

## 1. Imports & Paths

In [1]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

warnings.filterwarnings("ignore")

RAW_DATA_PATH = Path("../Dataset/clinical_data.csv")
OUT_DIR = Path("../Dataset/modelling")
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
N_SPLITS = 5
LABELS = ["NEV", "ACK", "SEK", "BCC", "SCC", "MEL"]
LABEL_TO_INT = {lbl: idx for idx, lbl in enumerate(LABELS)}
MALIGNANT_LABELS = {"BCC", "SCC", "MEL"}

## 2. Inspect Raw Dataset

In [2]:
raw_df = pd.read_csv(RAW_DATA_PATH)
print("Raw shape:", raw_df.shape)
print("Columns:", list(raw_df.columns))
print("\nDiagnostic counts:\n", raw_df["diagnostic"].astype(str).str.strip().str.upper().value_counts())

Raw shape: (2298, 26)
Columns: ['patient_id', 'lesion_id', 'smoke', 'drink', 'background_father', 'background_mother', 'age', 'pesticide', 'gender', 'skin_cancer_history', 'cancer_history', 'has_piped_water', 'has_sewage_system', 'fitspatrick', 'region', 'diameter_1', 'diameter_2', 'diagnostic', 'itch', 'grew', 'hurt', 'changed', 'bleed', 'elevation', 'img_id', 'biopsed']

Diagnostic counts:
 diagnostic
BCC    845
ACK    730
NEV    244
SEK    235
SCC    192
MEL     52
Name: count, dtype: int64


## 3. Schema Cleaning & Feature Engineering

In [3]:
def clean_clinical_schema(data: pd.DataFrame) -> pd.DataFrame:
    d = data.copy()

    d["diagnostic"] = d["diagnostic"].astype(str).str.strip().str.upper()
    d = d[d["diagnostic"].isin(LABELS)].reset_index(drop=True)
    d["y_multiclass"] = d["diagnostic"].map(LABEL_TO_INT).astype(int)
    d["is_cancer"] = d["diagnostic"].isin(MALIGNANT_LABELS).astype(int)

    # Pre-imputation missingness indicators
    q_markers = [c for c in ["fitspatrick", "diameter_1", "diameter_2", "gender"] if c in d.columns]
    d["has_questionnaire"] = d[q_markers].notna().any(axis=1).astype(int)

    for col in ["age", "fitspatrick", "diameter_1", "diameter_2"]:
        if col in d.columns:
            d[col] = pd.to_numeric(d[col], errors="coerce")

    d["diameter_missing"] = (d["diameter_1"].isna() | d["diameter_2"].isna()).astype(int)
    d["fitspatrick_missing"] = d["fitspatrick"].isna().astype(int)

    # Lesion size and aspect ratio proxies
    d1 = d["diameter_1"].clip(lower=0)
    d2 = d["diameter_2"].clip(lower=0)
    d["mean_diameter"] = 0.5 * (d1 + d2)
    d_max = np.maximum(d1, d2)
    d_min = np.minimum(d1, d2).replace(0, np.nan)
    d["diameter_ratio"] = np.where(
        d["diameter_1"].notna() & d["diameter_2"].notna(),
        (d_max / d_min).fillna(1.0),
        np.nan,
    )
    area_proxy = np.pi * (d1 / 2.0) * (d2 / 2.0)
    d["log_area"] = np.log1p(np.maximum(area_proxy, 0.0))

    # Keep TRUE, FALSE, and UNK distinct
    symptom_cols = [c for c in ["itch", "grew", "hurt", "changed", "bleed", "elevation"] if c in d.columns]
    true_cols, unk_cols = [], []
    for sc in symptom_cols:
        s_clean = d[sc].fillna("UNK").astype(str).str.strip().str.upper()
        d[f"{sc}_true"] = (s_clean == "TRUE").astype(int)
        d[f"{sc}_unk"] = (s_clean == "UNK").astype(int)
        true_cols.append(f"{sc}_true")
        unk_cols.append(f"{sc}_unk")
    d = d.drop(columns=symptom_cols)

    d["symptom_count"] = d[true_cols].sum(axis=1).astype(float)
    d["symptom_unk_count"] = d[unk_cols].sum(axis=1).astype(float)
    d["age_x_symptoms"] = d["age"] * d["symptom_count"]
    d["symptom_per_decade"] = d["symptom_count"] / np.maximum(d["age"] / 10.0, 1.0)

    q_bool_cols = [
        c for c in [
            "smoke", "drink", "pesticide", "skin_cancer_history",
            "cancer_history", "has_piped_water", "has_sewage_system"
        ] if c in d.columns
    ]
    for bc in q_bool_cols:
        s_str = d[bc].where(d[bc].notna(), "MISSING").astype(str).str.strip().str.upper()
        d[bc] = s_str.replace({"TRUE": "YES", "FALSE": "NO", "1": "YES", "0": "NO"})

    for bg_col in [c for c in ["background_father", "background_mother"] if c in d.columns]:
        s_bg = d[bg_col].fillna("MISSING").astype(str).str.strip().str.upper().replace({"BRASIL": "BRAZIL"})
        top_cats = {"POMERANIA", "GERMANY", "ITALY", "BRAZIL", "UNK", "MISSING"}
        d[bg_col] = np.where(s_bg.isin(top_cats), s_bg, "OTHER")

    for cat_c in [c for c in ["gender", "region"] if c in d.columns]:
        d[cat_c] = d[cat_c].fillna("MISSING").astype(str).str.strip().str.upper()

    cat_to_encode = [
        c for c in (q_bool_cols + ["gender", "region", "background_father", "background_mother"])
        if c in d.columns
    ]
    return pd.get_dummies(d, columns=cat_to_encode, dummy_na=False, dtype=int)


clean_df = clean_clinical_schema(raw_df)
drop_meta = {"patient_id", "lesion_id", "img_id", "diagnostic", "y_multiclass", "is_cancer"}
feature_cols = [c for c in clean_df.columns if c not in drop_meta]
print("Cleaned shape:", clean_df.shape, "| Feature count:", len(feature_cols))

Cleaned shape: (2298, 85) | Feature count: 79


## 4. Patient-Grouped Train/Test Split & Artifact Export

In [4]:
groups = clean_df["patient_id"].fillna(pd.Series(clean_df.index.astype(str), index=clean_df.index))
sgkf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
train_idx, test_idx = next(sgkf.split(clean_df, clean_df["y_multiclass"], groups))

train_df = clean_df.iloc[train_idx].reset_index(drop=True)
test_df = clean_df.iloc[test_idx].reset_index(drop=True)

overlap = set(train_df["patient_id"].dropna()) & set(test_df["patient_id"].dropna())
assert len(overlap) == 0, f"Patient leakage detected: {overlap}"

train_df.to_csv(OUT_DIR / "final_train.csv", index=False)
test_df.to_csv(OUT_DIR / "final_test.csv", index=False)
with open(OUT_DIR / "feature_metadata.json", "w") as f:
    json.dump({"feature_cols": feature_cols, "labels": LABELS, "label_to_int": LABEL_TO_INT}, f, indent=2)

print(f"Train shape: {train_df.shape} | Test shape: {test_df.shape} | Overlap: {len(overlap)}")
print("\nTrain class counts:\n", train_df["diagnostic"].value_counts().reindex(LABELS))
print("\nTest class counts:\n", test_df["diagnostic"].value_counts().reindex(LABELS))

Train shape: (1827, 85) | Test shape: (471, 85) | Overlap: 0

Train class counts:
 diagnostic
NEV    190
ACK    570
SEK    195
BCC    670
SCC    158
MEL     44
Name: count, dtype: int64

Test class counts:
 diagnostic
NEV     54
ACK    160
SEK     40
BCC    175
SCC     34
MEL      8
Name: count, dtype: int64
